# Understanding Visium Data and Running Deconvolution

In [1]:
%load_ext watermark
%watermark -a "Paul A. Wambo" -v -p numpy,pandas,matplotlib,scipy,seaborn,sklearn,scanpy,squidpy

Author: Paul A. Wambo

Python implementation: CPython
Python version       : 3.14.7
IPython version      : 9.17.1

numpy     : 2.4.6
pandas    : 3.0.6
matplotlib: 3.11.2
scipy     : 1.18.1
seaborn   : 0.13.2
sklearn   : 1.9.1
scanpy    : 1.12.4
squidpy   : 1.8.3



In [9]:
import scanpy as sc
import squidpy as sq

from pathlib import Path
import pandas as pd

In [11]:
path = Path(
    r"C:\Users\user\OneDrive\Desktop\GitHub\BioWorkbench\data"
    r"\Visium_HD_Tiny_3prime_Dataset_outs\binned_outputs\square_008um"
)

spatial_dir = path / "spatial"
positions_file = spatial_dir / "tissue_positions.parquet"
csv_file = spatial_dir / "tissue_positions.csv"

# Convert only if the expected CSV doesn't already exist.
if not csv_file.exists():
    positions = pd.read_parquet(positions_file)
    positions.to_csv(csv_file, index=False)

adata = sq.read.visium(
    path=path,
    counts_file="filtered_feature_bc_matrix.h5",
    load_images=True,
)

adata.var_names_make_unique()

adata = adata.copy()

adata

c:\Users\user\OneDrive\Desktop\GitHub\BioWorkbench\env\Lib\site-packages\scanpy\readwrite.py:248: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata = adata.copy()


AnnData object with n_obs × n_vars = 20830 × 32285
    obs: 'in_tissue', 'array_row', 'array_col'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'
    layers: None (.X)

In [12]:
adata.obs

,in_tissue,array_row,array_col
s_008um_00118_00105-1,1,118,105
s_008um_00165_00283-1,1,165,283
s_008um_00139_00261-1,1,139,261
s_008um_00185_00238-1,1,185,238
s_008um_00134_00304-1,1,134,304
...,...,...,...
s_008um_00169_00115-1,1,169,115
s_008um_00185_00033-1,1,185,33
s_008um_00121_00099-1,1,121,99
s_008um_00137_00209-1,1,137,209


In [13]:
adata.write("C:/Users/user/OneDrive/Desktop/GitHub/BioWorkbench/data/visium_human_colon.h5ad")

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]", regex=True)

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

In [ ]:
adata